# Drone Chase Tag: PPO eğitimi

1. **Çalışma zamanı → Çalışma zamanı türünü değiştir → A100 GPU** seç.
2. Hücreleri sırayla çalıştır. Sonuçlar Google Drive'a yazılır: oturum kapanırsa aynı hücreleri tekrar çalıştır,
   eğitim checkpoint'ten devam eder.
3. Eğitim bitince değerlendirme satırını ve son birkaç log satırını sohbete yapıştır; videoları Drive'dan indir.

**Arşiv kopyası:** `ppo-chaser-empty-nocrashrew-s0` koşusunda kullanılan notebook. Kod `3847d67` sürümüne sabitlendi; sonuçlar için `experiments/LOG.md`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!rm -rf /content/proje && git clone -q https://github.com/HasaNSaygili/DroneChaseTag.git /content/proje
%cd /content/proje
!git checkout -q 3847d67   # bu koşunun kod sürümü
!git log --oneline -1
!pip install -q "crazyflow[gpu]==0.4.0" -e .

## Koşu ayarları
`RUN` koşunun adı (experiments/LOG.md'deki adla aynı). `SET` config değişiklikleri; boş bırakılırsa varsayılanlar:
kovalayan öğrenir, rakip OPEN'ın kaçanı, boş arena, 16384 arena × 64 adım, toplam 500 M adım (~15 dk),
WCT kuralı (rakip çarparsa sen kazanırsın).

In [ ]:
RUN = "ppo-chaser-empty-nocrashrew-s0"
SET = "opp_crash_reward=0.0"   # doğrulama: rakibi çarptırmak ödül getirmez, tek kazanma yolu ebeleme
OUT = "/content/drive/MyDrive/DroneChaseTag/results"

In [ ]:
%cd /content/proje
!python -W ignore scripts/train.py --name {RUN} --out {OUT} {('--set ' + SET) if SET else ''} 2>&1 | grep -v -E "warp|spec|fromstring"

## Değerlendirme ve video
Hiç görülmemiş arenalarda, gürültüsüz oynatır; 3 maçı videoya çeker (Drive'da `videos/`).

In [ ]:
%cd /content/proje
!MUJOCO_GL=egl python -W ignore scripts/watch_policy.py --run {OUT}/{RUN} --games 3 --eval-worlds 1024 --out /content/drive/MyDrive/DroneChaseTag/videos 2>&1 | grep -v -E "warp|spec|fromstring"

## Eğitim eğrisi

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
rows = [json.loads(l) for l in open(f"{OUT}/{RUN}/metrics.jsonl")]
steps = np.array([r['steps'] for r in rows]) / 1e6
C = np.array([r['outcome_counts'] for r in rows]); tot = C[:, 1:].sum(1, keepdims=True).clip(1)
names = ['ebeleme', 'süre doldu', 'kaçan çarptı', 'kovalayan çarptı']
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for i, n in enumerate(names):
    ax[0].plot(steps, np.convolve(C[:, i + 1] / tot[:, 0], np.ones(5) / 5, 'same'), label=n)
ax[0].set_title('sonuç oranları'); ax[0].set_xlabel('M adım'); ax[0].legend()
ax[1].plot(steps, [r['ep_len'] / 50 for r in rows]); ax[1].set_title('ortalama maç süresi (s)')
ax[2].plot(steps, [r['explained_var'] for r in rows], label='explained var')
ax[2].plot(steps, [r['std'] for r in rows], label='std'); ax[2].legend(); ax[2].set_title('critic ve keşif')
plt.show()